# BUSI MedSAM: 300-image pilot
Compare pretrained MedSAM with decoder-only adaptation. Select **Runtime → Change runtime type → GPU**. This notebook has not yet been executed with real BUSI data; the source includes synthetic and official-architecture tests. See README for scope and citations.

Use a trusted original BUSI archive and the official `medsam_vit_b.pth` checkpoint. Evaluation uses annotation-derived boxes, not automatic detection. Keep the test set untouched during model selection.


In [ ]:
from pathlib import Path
import sys, subprocess, zipfile
REPO = Path('/content/medsam-busi-pilot')
# Clone the published source. Set USE_ZIP=True to use a supplied source ZIP.
USE_ZIP = False
if not (REPO / 'pyproject.toml').exists() and not USE_ZIP:
    subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/Ayorindealase/medsam-busi-pilot.git', str(REPO)], check=True)
if not (REPO / 'pyproject.toml').exists():
    from google.colab import files
    uploaded = files.upload()
    archive = next(name for name in uploaded if name.endswith('.zip'))
    with zipfile.ZipFile(archive) as z:
        # Validate paths before extracting an archive you trust.
        for item in z.infolist():
            target = (Path('/content') / item.filename).resolve()
            if not target.is_relative_to(Path('/content')):
                raise ValueError('Unsafe archive path')
        z.extractall('/content')
assert (REPO / 'pyproject.toml').exists(), 'ZIP must contain medsam-busi-pilot/'
import os
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt'], check=True)


If NumPy was already imported before installation and an import error occurs, restart the runtime, rerun the setup cell to restore its variables, and then continue. The notebook retains Colab’s installed torch/torchvision CUDA pair.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'
print(torch.__version__, torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Edit these paths. Extract BUSI in Drive before running this cell.
DATA = Path('/content/drive/MyDrive/MedSAM/Dataset_BUSI_with_GT')
CHECKPOINT = Path('/content/drive/MyDrive/MedSAM/medsam_vit_b.pth')
WORK = Path('/content/drive/MyDrive/MedSAM/pilot_300_seed2026')
RUN_NAME = 'trial_01'  # Use a new name for a fresh training run.
assert (DATA / 'benign').is_dir() and (DATA / 'malignant').is_dir()
assert CHECKPOINT.is_file(), 'Download MedSAM ViT-B weights from the official README'
WORK.mkdir(parents=True, exist_ok=True)
MANIFEST = WORK / 'manifest.json'
CACHE = WORK / 'cache'
RUN = WORK / RUN_NAME
RUN.mkdir(parents=True, exist_ok=True)
def run(*args):
    subprocess.run([sys.executable, '-m', 'medsam_busi.cli', *map(str, args)], check=True)
common = ['--root', DATA, '--manifest', MANIFEST, '--checkpoint', CHECKPOINT]
run('prepare', '--root', DATA, '--output', MANIFEST, '--size', 300, '--seed', 2026)
import json
from collections import Counter
manifest = json.loads(MANIFEST.read_text())
print(Counter((r['split'], r['label']) for r in manifest['records']))
print('Exclusions:', manifest['excluded'])


Inspect masks and prompts before training. This preview uses training images only. If the archive or protocol changes, use a new work folder rather than overwriting the established split.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from medsam_busi.data import read_pair, box_from_mask
training = [r for r in manifest['records'] if r['split'] == 'train']
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, row in zip(axes, training[:3]):
    image, mask = read_pair(DATA, row)
    ax.imshow(image)
    ax.imshow(mask, alpha=.25, cmap='Reds')
    x0, y0, x1, y1 = box_from_mask(mask)
    ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fill=False, edgecolor='yellow'))
    ax.set_title(row['id']); ax.axis('off')
plt.show()


Cache frozen image embeddings once. This is the main encoder workload and requires roughly 1.2 GiB for cache files. Existing cache entries are reused only with matching dataset, preprocessing and checkpoint identity.

In [ ]:
run('cache', *common, '--output', CACHE)


Train the decoder, selecting the best epoch by validation Dice. Existing training output folders are protected; change RUN_NAME to run a new trial. Training restart/resume is not implemented; embedding caching is resumable.

In [ ]:
run('train', *common, '--cache', CACHE, '--output', RUN / 'train', '--epochs', 10, '--lr', 1e-4, '--accumulation', 4, '--seed', 2026)


In [ ]:
history = json.loads((RUN / 'train/history.json').read_text())
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot([r['epoch'] for r in history], [r['train_loss'] for r in history]); axes[0].set_title('Train loss')
axes[1].plot([r['epoch'] for r in history], [r['val_dice'] for r in history]); axes[1].set_title('Validation Dice')
plt.show()


Final evaluation: run only after fixing all hyperparameters. Both arms use the same held-out test images, tight ground-truth boxes, original-resolution scoring and threshold. No score from this section should guide further tuning.

In [ ]:
run('evaluate', *common, '--cache', CACHE, '--output', RUN / 'baseline')
run('evaluate', *common, '--cache', CACHE, '--decoder', RUN / 'train/best_decoder.pt', '--output', RUN / 'tuned')
run('compare', '--baseline-path', RUN / 'baseline/metrics.json', '--tuned-path', RUN / 'tuned/metrics.json', '--output', RUN / 'comparison.json')
comparison = json.loads((RUN / 'comparison.json').read_text())
print(json.dumps(comparison, indent=2))


In [ ]:
from PIL import Image
row = next(r for r in manifest['records'] if r['split'] == 'test')
image, mask = read_pair(DATA, row)
name = row['id'].replace('/', '_') + '.png'
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, title, overlay in zip(axes, ['Ground truth', 'Baseline', 'Fine-tuned'], [mask, Image.open(RUN/'baseline/predictions'/name), Image.open(RUN/'tuned/predictions'/name)]):
    ax.imshow(image); ax.imshow(overlay, alpha=.3, cmap='Reds'); ax.set_title(title); ax.axis('off')
plt.show()


Export provenance and numerical results for review. This archive excludes raw images, predictions and model weights; those stay in Drive. The image bootstrap interval is exploratory and does not establish patient-independent generalization.

In [ ]:
report_zip = RUN / 'pilot_reports.zip'
with zipfile.ZipFile(report_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(MANIFEST, 'manifest.json')
    z.write(CACHE / 'metadata.json', 'cache_metadata.json')
    for relative in ['comparison.json', 'train/run.json', 'train/history.json', 'baseline/metrics.json', 'baseline/per_image.csv', 'tuned/metrics.json', 'tuned/per_image.csv']:
        z.write(RUN / relative, relative)
from google.colab import files
files.download(str(report_zip))
